# 20 — Tolerance Intervals / 容差区间

**Chapter 20 — File 1 of 2**

## Summary / 摘要

A tolerance interval is a statistical range that contains a specified proportion of the population with a given confidence level. Unlike confidence intervals (which estimate parameters) and prediction intervals (which estimate single future values), tolerance intervals bound the range where a proportion p of the population is expected to fall with confidence level (1-alpha). For normally distributed data with 95% coverage and 99% confidence, the interval uses chi-square and normal critical values.

容差区间是一个统计范围，它以给定的置信水平包含总体的指定比例。与置信区间（估计参数）和预测区间（估计单个未来值）不同，容差区间界定了其中总体比例p预计以置信水平(1-alpha)落在的范围。对于正态分布的数据，具有95%覆盖率和99%置信度，区间使用卡方和正态临界值。

## Step 1 — Import Libraries / 导入库

In [ ]:
# Import required libraries
# 导入所需库
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt

# Set random seed for reproducibility
# 设置随机种子以保证可重复性
np.random.seed(42)

## Step 2 — Generate Normal Distribution Data / 生成正态分布数据

In [ ]:
# Generate data from normal distribution
# 从正态分布生成数据
mu = 100  # Mean / 均值
sigma = 15  # Standard deviation / 标准差
n = 50  # Sample size / 样本大小

data = np.random.normal(mu, sigma, n)

# Calculate sample statistics
# 计算样本统计量
sample_mean = np.mean(data)
sample_std = np.std(data, ddof=1)  # Use unbiased estimator / 使用无偏估计

# Display data properties
# 显示数据属性
print(f"Sample size: {n}")
print(f"Sample mean: {sample_mean:.2f}")
print(f"Sample std: {sample_std:.2f}")
print(f"Min value: {np.min(data):.2f}")
print(f"Max value: {np.max(data):.2f}")

## Step 3 — Define Tolerance Interval Parameters / 定义容差区间参数

In [ ]:
# Tolerance interval parameters
# 容差区间参数
coverage = 0.95  # Proportion of population to cover / 要覆盖的总体比例
confidence = 0.99  # Confidence level / 置信水平
alpha = 1 - confidence  # Significance level / 显著性水平

# Compute critical values
# 计算临界值
# z_coverage: normal quantile for coverage level
# z_coverage: 覆盖水平的正态分位数
z_coverage = stats.norm.ppf((1 + coverage) / 2)

# chi2_val: chi-square quantile for confidence level
# chi2_val: 置信水平的卡方分位数
chi2_val = stats.chi2.ppf(1 - alpha, df=n-1)

# k_factor: correction factor combining both values
# k_factor: 组合两个值的修正因子
k_factor = np.sqrt((n - 1) * (1 + 1/n) * chi2_val / stats.chi2.ppf(coverage, df=1))

print(f"Coverage: {coverage*100:.1f}%")
print(f"Confidence: {confidence*100:.1f}%")
print(f"z_coverage: {z_coverage:.4f}")
print(f"chi2_val: {chi2_val:.4f}")
print(f"k_factor: {k_factor:.4f}")

## Step 4 — Calculate Tolerance Interval / 计算容差区间

In [ ]:
# Calculate margin of error using k-factor and sample standard deviation
# 使用k因子和样本标准差计算误差范围
margin = z_coverage * sample_std * np.sqrt(1 + 1/n)

# Calculate tolerance interval bounds
# 计算容差区间边界
lower_bound = sample_mean - margin
upper_bound = sample_mean + margin

# Display tolerance interval
# 显示容差区间
print(f"\nTolerance Interval:")
print(f"Lower bound: {lower_bound:.2f}")
print(f"Upper bound: {upper_bound:.2f}")
print(f"Interval width: {upper_bound - lower_bound:.2f}")
print(f"\nInterpretation: We are {confidence*100:.0f}% confident that ")
print(f"{coverage*100:.0f}% of the population lies between {lower_bound:.2f} and {upper_bound:.2f}")

## Step 5 — Visualize Data and Interval / 可视化数据和区间

In [ ]:
# Create visualization
# 创建可视化
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Plot 1: Histogram with tolerance interval
# 图1: 直方图与容差区间
axes[0].hist(data, bins=15, edgecolor='black', alpha=0.7, density=True)

# Overlay normal distribution curve
# 覆盖正态分布曲线
x = np.linspace(data.min() - 10, data.max() + 10, 100)
axes[0].plot(x, stats.norm.pdf(x, sample_mean, sample_std), 'r-', linewidth=2, label='Normal fit')

# Add tolerance interval lines
# 添加容差区间线
axes[0].axvline(lower_bound, color='green', linestyle='--', linewidth=2, label='Tolerance interval')
axes[0].axvline(upper_bound, color='green', linestyle='--', linewidth=2)
axes[0].axvline(sample_mean, color='blue', linestyle='-', linewidth=2, label='Sample mean')

axes[0].set_xlabel('Value')
axes[0].set_ylabel('Density')
axes[0].set_title(f'Data Distribution with {coverage*100:.0f}% / {confidence*100:.0f}% Tolerance Interval')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot 2: Box plot with tolerance interval
# 图2: 箱线图与容差区间
axes[1].boxplot([data], vert=True, labels=['Data'])
axes[1].axhline(lower_bound, color='green', linestyle='--', linewidth=2, label='Tolerance interval')
axes[1].axhline(upper_bound, color='green', linestyle='--', linewidth=2)
axes[1].axhline(sample_mean, color='blue', linestyle='-', linewidth=2, label='Sample mean')

axes[1].set_ylabel('Value')
axes[1].set_title('Box Plot with Tolerance Interval')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Learning Notes / 学习笔记

- **Statistical Concept**: Tolerance intervals are wider than confidence intervals because they account for both estimation uncertainty (from sample variability) and the natural spread of the population. The k-factor multiplies the standard deviation to capture a specified population proportion at a chosen confidence level, combining chi-square and normal distributions.
  
  **统计概念**: 容差区间比置信区间更宽，因为它们考虑估计不确定性（来自样本变异性）和总体的自然分布。k因子将标准差相乘以在选定的置信水平处捕获指定的总体比例，结合了卡方和正态分布。

- **ML Application**: Tolerance intervals are critical in quality control, manufacturing process monitoring, and risk assessment. They define specification limits in production systems, ensure product compliance with standards, and help identify when processes drift outside acceptable ranges. Essential for setting operational thresholds in production ML systems.
  
  **ML应用**: 容差区间在质量控制、制造过程监控和风险评估中至关重要。它们定义了生产系统中的规格限制，确保产品符合标准，并帮助识别流程何时超出可接受范围。对于在生产ML系统中设置操作阈值至关重要。

➡️ **Next**: `02_tolerance_to_sample_size.ipynb`

## Complete Code / 完整代码一览

In [ ]:
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt

np.random.seed(42)
mu, sigma, n = 100, 15, 50
data = np.random.normal(mu, sigma, n)

sample_mean = np.mean(data)
sample_std = np.std(data, ddof=1)

coverage, confidence = 0.95, 0.99
alpha = 1 - confidence

z_coverage = stats.norm.ppf((1 + coverage) / 2)
chi2_val = stats.chi2.ppf(1 - alpha, df=n-1)

margin = z_coverage * sample_std * np.sqrt(1 + 1/n)
lower_bound = sample_mean - margin
upper_bound = sample_mean + margin

print(f"Tolerance Interval: [{lower_bound:.2f}, {upper_bound:.2f}]")
print(f"Width: {upper_bound - lower_bound:.2f}")